# Feature selection
I current have a data file for 5 drugs and around 41k features. We need to analyse and build drug response models for the chosen drugs. Instead of doing feature selection on the genes, I have opted to utilise pathway scores. Chosen based on this [Nature paper](https://www.nature.com/articles/s41467-022-33291-z.pdf). 

This paper utilised pathway enrichment scores calculated using log normalised TPM scores. Pathway scores are more generalizable and can be extended to other analysis easily. Single genes however may prove to be noisy at such a small scale.

I will calculate the scores using GSVA.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import data, models, train, results_store

pathway_enriched_data = data.build_pathway_enriched_data(
    required_data_path="../data/required_data.parquet",
    gsva_scores_path="../data/gsva_C2_scores.csv",
)
pathway_enriched_data.to_csv("../data/pathway_enriched_scores_C2.csv", index=False)
pathway_enriched_data.shape

## Train-Test-Val Split
SPlit by drug and by cell line. Develop one model for each drug.

In [9]:
train_df, val_df, test_df = data.make_splits(pathway_enriched_data)
train_df.shape, val_df.shape, test_df.shape

((3263, 210), (465, 210), (935, 210))

In [10]:
data.save_splits(train_df, val_df, test_df, outdir="../data/test", suffix="C2")

## Model Training
MLP, XGBoost, LightGBm, Random Forest (?), Decision Trees, SVM

In [2]:
train_df, val_df, test_df = data.load_splits("../data/test", suffix="hallmark")
train_df.shape, val_df.shape, test_df.shape

((3263, 64), (465, 64), (935, 64))

In [3]:
baseline_results_df, baseline_fitted, baseline_run_id = train.run_stage(
    models.baseline_models(), train_df, val_df, stage="baseline",feature_prefix="HALLMARK_"
)
print("run_id:", baseline_run_id)
baseline_results_df

run_id: 38b7f6c0


,feature_prefix,DRUG_ID,DRUG_NAME,model,n_train,n_val,n_trees_used,rmse,r2
0,HALLMARK_,1010,Gefitinib,elasticnet,653,93,None,0.984798,0.185111
1,HALLMARK_,1010,Gefitinib,dummy_regressor,653,93,None,1.091717,-0.001438
2,HALLMARK_,1080,Paclitaxel,elasticnet,653,93,None,1.602959,0.200358
3,HALLMARK_,1080,Paclitaxel,dummy_regressor,653,93,None,1.794861,-0.002565
4,HALLMARK_,1168,Erlotinib,elasticnet,651,93,None,1.323532,0.091334
5,HALLMARK_,1168,Erlotinib,dummy_regressor,651,93,None,1.394079,-0.008115
6,HALLMARK_,1190,Gemcitabine,elasticnet,654,93,None,2.365554,0.344350
7,HALLMARK_,1190,Gemcitabine,dummy_regressor,654,93,None,2.927124,-0.003895
8,HALLMARK_,1199,Tamoxifen,elasticnet,652,93,None,1.108893,0.305548
9,HALLMARK_,1199,Tamoxifen,dummy_regressor,652,93,None,1.330686,-0.000033


In [4]:
candidate_results_df, candidate_fitted, candidate_run_id = train.run_stage(
    models.candidate_models(), train_df, val_df, stage="candidates", feature_prefix="HALLMARK_"
)
print("run_id:", candidate_run_id)
candidate_results_df

run_id: 84d9fa7f


,feature_prefix,DRUG_ID,DRUG_NAME,model,n_train,n_val,n_trees_used,rmse,r2
2,HALLMARK_,1010,Gefitinib,svm,653,93,NaN,0.957251,0.230062
1,HALLMARK_,1010,Gefitinib,random_forest,653,93,NaN,0.974658,0.201806
4,HALLMARK_,1010,Gefitinib,xgboost,653,93,59.0,0.998309,0.162599
5,HALLMARK_,1010,Gefitinib,lightgbm,653,93,78.0,1.007408,0.147265
0,HALLMARK_,1010,Gefitinib,decision_tree,653,93,NaN,1.124696,-0.062854
3,HALLMARK_,1010,Gefitinib,mlp,653,93,NaN,1.163666,-0.137786
9,HALLMARK_,1080,Paclitaxel,mlp,653,93,NaN,1.493763,0.305593
11,HALLMARK_,1080,Paclitaxel,lightgbm,653,93,130.0,1.552229,0.250170
10,HALLMARK_,1080,Paclitaxel,xgboost,653,93,99.0,1.567363,0.235477
7,HALLMARK_,1080,Paclitaxel,random_forest,653,93,NaN,1.582732,0.220411


In [15]:
df =results_store.load_results()

In [17]:
required_df = df[df['feature_prefix']=='PID_'].drop(columns=['timestamp','run_id','stage'])

In [18]:
results_store.best_models(required_df)

,timestamp,feature_prefix,DRUG_ID,DRUG_NAME,model,n_train,n_val,n_trees_used,rmse,r2
12,2026-07-11T08:28:02.865771+00:00,PID_,1010,Gefitinib,svm,653,93,NaN,0.937700,0.261192
17,2026-07-11T08:28:02.865771+00:00,PID_,1080,Paclitaxel,random_forest,653,93,NaN,1.545131,0.257012
27,2026-07-11T08:28:02.865771+00:00,PID_,1168,Erlotinib,lightgbm,651,93,144.0,1.256775,0.180685
31,2026-07-11T08:28:02.865771+00:00,PID_,1190,Gemcitabine,mlp,654,93,NaN,2.260489,0.401298
8,2026-07-11T08:28:02.865771+00:00,PID_,1199,Tamoxifen,elasticnet,652,93,NaN,1.108090,0.306553


In [6]:
all_dfs = pd.concat([baseline_results_df, candidate_results_df],ignore_index=True)
results_store.best_models(all_dfs)

,timestamp,feature_prefix,DRUG_ID,DRUG_NAME,model,n_train,n_val,n_trees_used,rmse,r2
10,2026-07-11T08:24:02.832631+00:00,HALLMARK_,1010,Gefitinib,svm,653,93,NaN,0.957251,0.230062
16,2026-07-11T08:24:02.832631+00:00,HALLMARK_,1080,Paclitaxel,mlp,653,93,NaN,1.493763,0.305593
22,2026-07-11T08:24:02.832631+00:00,HALLMARK_,1168,Erlotinib,xgboost,651,93,150.0,1.307578,0.113108
6,2026-07-11T08:24:02.832631+00:00,HALLMARK_,1190,Gemcitabine,elasticnet,654,93,None,2.365554,0.344350
8,2026-07-11T08:24:02.832631+00:00,HALLMARK_,1199,Tamoxifen,elasticnet,652,93,None,1.108893,0.305548
